In [1]:
import os 
import pathlib as Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'

def load_data_from_path(folder_path):
    examples = []
    for label in os.lisrdir(folder_path):
        full_path = os.path.join(folder_path, label)
        for file_name in os.listdir(full_path):
            file_path = os.path.join(full_path, file_name)

            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            sentence = " ".join(lines)

            if label == "neg":
                label = 0
            if label == "pos":
                label = 1

            data = {
                'setence': sentence,
                'label': label
            }

            examples.append(data)

    return pd.DataFrame(examples)


folder_paths = {
    'train': DATA_DIR / 'train',
    'valid': DATA_DIR / 'test',
    'test': DATA_DIR / 'test'
}

train_df = load_data_from_path(folder_path['train'])
valid_df = load_data_from_path(folder_path['valid'])
test_df = load_data_from_path(folder_path['test'])

ModuleNotFoundError: No module named 'pandas'

In [2]:
from langid.langid import LanguageIdentifier, model

def identifiy_vn(df):
    identifier = LanguageIdentifier.from_modelstring(
        model, norm_probs=True
    )
    not_vi_idx = set()
    THRESHOLD = 0.9

    for idx, row in df.iterrows():
        score = identifier.classify(row['sentence'])
        if score[0] != "vi" or (score[0] == "vi" and score[1] <= THRESHOLD):
            not_vi_idx.add(idx)

        vi_df = df[~df.index.isin(not_vi_idx)]
        not_vi_df = df[df.index.isin(not_vi_idx)]

        return vi_df, not_vi_df

train_df_vi, train_df_other = identifiy_vn(train_df)

ModuleNotFoundError: No module named 'langid'

In [3]:
import re
import string

def preprocess_text(text):

    url_pattern = re.compile(r'https?://\s+\wwww\.\s+')
    text = url_pattern.sub(r" ", text)

    html_pattern = re.compile(r'<[^<>]+>')
    text = html_pattern.sub(" ", text)

    replace_chars = list(string.punctuation + string.digits)
    for char in replace_chars:
        text = text.replace(char, " ")

    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        "]+", flags=re.UNICODE)

    text = emoji_pattern.sub(r" ", text)

    text = " ".join(text.split())

    return text.lower()

train_df_vi['preprocess_sentence'] = [
    preprocess_text(row["sentence"]) 
    for _, row in train_df_vi.iterrows()
]

valid_df['preprocess_sentence'] = [
    preprocess_text(row['sentence'])
    for _, row in valid_df.iterrow()
]

test_df['preprocess_sentence'] = [
    preprocess_text(row['sentence'])
    for _, row in test_df.iterrow()
]

NameError: name 'train_df_vi' is not defined

In [4]:
from torchtext.data.utils import get_tokenizer
tokenizer = get_tokenizer("basic_english")

def yield_tokens(sentences, tokenizer):
    for sentence in sentences:
        yield tokenizer(sentence)

from torchtext.vocab import buil_vocab_from_iterator
vocab_size = 1000
vocabulary = buil_vocab_from_iterator(
    yield_tokens(train_df_vi['preprocess_sentence'], tokenizer),
    max_tokens = vocab_size,
    spectials=["<pad>", "<unk>"]
)
vocabulary.set_default_index(vocabulary["<unk>"])

from torchtext.data.functional import to_map_style_dataset
def prepare_dataset(df):
    for idx, row in df.iterrow():
        sentence = row['preprocess_sentence']
        encoded_sentence = vocabulary(tokenizer(sentence))
        label = row['label']
        yield encoded_sentence, label

train_dataset = prepare_dataset(train_df_vi)
train_dataset = to_map_style_dataset(train_dataset)

valid_dataset = prepare_dataset(valid_df)
valid_dataset = to_map_style_dataset(valid_dataset)

ModuleNotFoundError: No module named 'torchtext'

In [5]:
from torch.utils.data import DataLoader

def collate_batch(batch):
    encoded_sentences, labels = [], []

    for encoded_sentence, label in batch:
        labels.append(label)
        encoded_sentence = torch.tensor(encoded_sentence, dtype=torch.int64)

        encoded_sentences.append(encoded_sentence)

    labels = torch.tensor(labels, dtype=torch.int64)
    encoded_sentences = pad.sequence(
        encoded_sentences,
        padding_value = vocabulary["<pad>"]
    )

    return encoded_sentences, labels

batch_size = 128

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_batch
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch
)

NameError: name 'train_dataset' is not defined

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, kernel_sizes,
                 num_filters, num_classes):
        super(TextCNN, self).__init__()

        self.vocab_size=vocab_size,
        self.embedding_dim=embeding_dim,
        self.kernel_sizes=kernel_sizes,
        self.num_filters=num_filters,
        self.num_classes=num_classes,
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.conv = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k,
                stride=1
            ) for k in kernel_sizes])
        self.fc = nn.Linear(len(kernel_sizes) * num_filters, num_classes)

    def forward(self, x):
        batch_size, sequence_length = x.shape
        x = self.embedding(x.T).transpose(1,2)
        x = [F.relu(conv(x)) for conv in self.conv]
        x = [F.max_pool1d(c, c.size(-1)).squeeze(dim=-1) for c in x]
        x = torch.cat(x, dim=1)
        x = self.fc(x)

        return x